# **Setup & Installation**

In [1]:
%pip install langchain-community pypdf langchain-huggingface sentence-transformers

# **Documents and document loaders**

In [2]:
!curl -L -o Bert_paper.pdf "https://arxiv.org/pdf/1810.04805.pdf"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   217  100   217    0     0   1358      0 --:--:-- --:--:-- --:--:--  1364
100  756k  100  756k    0     0  2833k      0 --:--:-- --:--:-- --:--:-- 2833k


In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF into LangChain Document objects
file_path = "Bert_paper.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages.")
print(f"Preview of page 1:\n{docs[0].page_content[:200]}\n")
print(f"Metadata:\n{docs[0].metadata}")

Loaded 16 pages.
Preview of page 1:
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout

Metadata:
{'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Bert_paper.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


# **Splitting**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the document to ensure meanings aren't "washed out"
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(f"Split the document into {len(all_splits)} chunks.")

Split the document into 83 chunks.


# **Embeddings**

In [6]:
%pip install langchain-huggingface sentence-transformers

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Sanity check: verify the embeddings generate the correct vector lengths
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated vectors of length 768

[0.0052108620293438435, 0.027721881866455078, 0.001389949582517147, 0.08294462412595749, 0.001529207220301032, 0.025097651407122612, 0.008785158395767212, -0.04010576382279396, -0.02537068910896778, -0.051830556243658066]


# **Vector stores**

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

# Create the vector store and add the embedded document chunks
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print("Documents successfully added to the vector store!")

Documents successfully added to the vector store!


In [9]:
results = vector_store.similarity_search(
    "What does BERT stand for?"
)

print(results[0])

page_content='BERT including:
– Effect of Number of Training Steps; and
– Ablation for Different Masking Proce-
dures.
A Additional Details for BERT
A.1 Illustration of the Pre-training Tasks
We provide examples of the pre-training tasks in
the following.
Masked LM and the Masking Procedure As-
suming the unlabeled sentence is my dog is
hairy, and during the random masking procedure
we chose the 4-th token (which corresponding to
hairy), our masking procedure can be further il-
lustrated by
• 80% of the time: Replace the word with the
[MASK] token, e.g., my dog is hairy →
my dog is [MASK]
• 10% of the time: Replace the word with a
random word, e.g., my dog is hairy → my
dog is apple
• 10% of the time: Keep the word un-
changed, e.g., my dog is hairy → my dog
is hairy . The purpose of this is to bias the
representation towards the actual observed
word.
The advantage of this procedure is that the
Transformer encoder does not know which words
it will be asked to predict or which have been

Async query:

In [10]:
# Asynchronous similarity search
results = await vector_store.asimilarity_search("What is the Masked Language Model (MLM)?")

print(results[0])

page_content='almost immediately.
C.2 Ablation for Different Masking
Procedures
In Section 3.1, we mention that BERT uses a
mixed strategy for masking the target tokens when
pre-training with the masked language model
(MLM) objective. The following is an ablation
study to evaluate the effect of different masking
strategies.
200 400 600 800 1,000
76
78
80
82
84
Pre-training Steps (Thousands)
MNLI Dev Accuracy
BERTBASE (Masked LM)
BERTBASE (Left-to-Right)
Figure 5: Ablation over number of training steps. This
shows the MNLI accuracy after ﬁne-tuning, starting
from model parameters that have been pre-trained for
k steps. The x-axis is the value ofk.
Note that the purpose of the masking strategies
is to reduce the mismatch between pre-training
and ﬁne-tuning, as the [MASK] symbol never ap-
pears during the ﬁne-tuning stage. We report the
Dev results for both MNLI and NER. For NER,
we report both ﬁne-tuning and feature-based ap-
proaches, as we expect the mismatch will be am-' metadata={'pr

Return scores:

In [11]:
# Search returning distance/similarity scores
results = vector_store.similarity_search_with_score("How does BERT use bidirectional context?")
doc, score = results[0]

print(f"Score: {score}\n")
print(doc)

Score: 0.6805638822694366

page_content='Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
is presented in the left part of Figure 1.
Task #1: Masked LM Intuitively, it is reason-
able to believe that a deep bidirectional model is
strictly more powerful than either a left-to-right
model or the shallow concatenation of a left-to-
right and a right-to-left model. Unfortunately,
standard conditional language models can only be
trained left-to-right or right-to-left, since bidirec-
tional conditioning would allow each word to in-
directly “see itself”, and the model could trivially
predict the target word in a multi-layered context.
former is often referred to as a “Transformer encoder” while
the left-context-only version is referred to as a “Transformer
decoder” since it can be used for text generation.
In order to train a deep bidirectional representa-
tion, we simply mask some percentage of the input
tokens at random, and then predict those 

Return documents based on similarity to an embedded query:

In [12]:
# Embed a query first, then search using that vector
embedding = embeddings.embed_query("What are the advantages of BERT over OpenAI GPT?")
results = vector_store.similarity_search_by_vector(embedding)

print(results[0])

page_content='size as OpenAI GPT for comparison purposes.
Critically, however, the BERT Transformer uses
bidirectional self-attention, while the GPT Trans-
former uses constrained self-attention where every
token can only attend to context to its left.4
1https://github.com/tensorﬂow/tensor2tensor
2http://nlp.seas.harvard.edu/2018/04/03/attention.html
3In all cases we set the feed-forward/ﬁlter size to be 4H,
i.e., 3072 for the H = 768 and 4096 for the H = 1024.
4We note that in the literature the bidirectional Trans-' metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Bert_paper.pdf', 'total_pages': 16, 'page': 2, 'page_label': '3', 'start_index': 3195}


# **Retrievers**

In [15]:
# Convert the vector store into a retriever abstraction and run a batch query
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

results = retriever.batch(
    [
        "What are the two pre-training tasks used for BERT?",
        "What is Next Sentence Prediction (NSP)?",
    ],
)

print(results)

[[Document(id='8937a8ed-2cca-4ea6-96ab-2878b9e209a0', metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Bert_paper.pdf', 'total_pages': 16, 'page': 11, 'page_label': '1', 'start_index': 2380}, page_content='BERT including:\n– Effect of Number of Training Steps; and\n– Ablation for Different Masking Proce-\ndures.\nA Additional Details for BERT\nA.1 Illustration of the Pre-training Tasks\nWe provide examples of the pre-training tasks in\nthe following.\nMasked LM and the Masking Procedure As-\nsuming the unlabeled sentence is my dog is\nhairy, and during the random masking procedure\nwe chose the 4-th token (which corresponding to\nhairy), our masking procedure can be further i